# 🎙️ Fine-tuning Thonburian Whisper with Data Cleaning & Pseudo-Labels

This notebook walks through a complete pipeline for **domain-adaptive fine-tuning** of [Thonburian Whisper](https://github.com/biodatlab/thonburian-whisper) — an already strong Thai ASR model, using:

1. **Data cleaning** — duration filtering, SNR-based noise filtering, text normalization
2. **Pseudo-label generation** — using the teacher model to label additional data, filtered by confidence
3. **Fine-tuning** — Low-Rank Adaptive Fine-tuning fine-tuning with BF16 mixed precision (RTX 3090 compatible)
4. **Evaluation** — Thai-aware WER using deepcut tokenizer

### 📋 Table of Contents

0. [Setup & Installation](#0-setup)
1. [Load Base Model & Processor](#1-load-model)
2. [Load & Explore Dataset](#2-load-data)
3. [Data Cleaning Pipeline](#3-data-cleaning)
4. [Pseudo-Label Generation](#4-pseudo-labels)
5. [Prepare Data for Training](#5-prepare-training)
6. [Fine-tuning](#6-finetuning)
7. [Evaluation](#7-evaluation)
8. [Inference & Export](#8-inference)

---
## 0. Setup & Installation <a id='0-setup'></a>

In [2]:
!pip install -q transformers datasets accelerate peft bitsandbytes
!pip install -q librosa soundfile jiwer evaluate
!pip install -q pythainlp deepcut

In [3]:
import os
import string
import numpy as np
import pandas as pd
import torch
import librosa
import evaluate
from tqdm.auto import tqdm
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Audio, Dataset, concatenate_datasets
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    WhisperTokenizer,
    WhisperFeatureExtractor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

from jiwer import process_words, wer_default
from pythainlp.tokenize import word_tokenize as thai_tokenize

# Check GPU
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/home/badboy-005/anaconda3/envs/alveolar_project/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 3090
VRAM: 25.3 GB


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# For Thai fine-tuning, you can switch to:
#   "biodatlab/whisper-th-small-combined"     — Requires HF token
#   "biodatlab/whisper-th-medium-combined"    — Requires HF token

MODEL_NAME = "biodatlab/whisper-th-small-combined"  
LANGUAGE = "th"
TASK = "transcribe"
SAMPLING_RATE = 16_000

# HuggingFace token — needed to download gated models & datasets
# Get yours at: https://huggingface.co/settings/tokens
HF_TOKEN = ''  # Set to your token string if using gated models

# Data cleaning thresholds
MIN_DURATION_SEC = 1.0    # Discard clips shorter than this
MAX_DURATION_SEC = 30.0   # Discard clips longer than this
MIN_SNR_DB = 10.0         # Discard clips with SNR below this

# Pseudo-label filtering - lower threshold for Khummuang dialect
PSEUDO_LABEL_CONFIDENCE_THRESHOLD = 0.3  # Lowered to get more pseudo-labels

# Training
OUTPUT_DIR = "./whisper-th-finetuned"
BATCH_SIZE = 8            # Fits on T4 with LoRA
GRAD_ACCUM_STEPS = 2      # Effective batch size = 16
LEARNING_RATE = 1e-4
NUM_TRAIN_EPOCHS = 5      # Increased for better training
WARMUP_STEPS = 50
MAX_TRAIN_SAMPLES = None  # Use all training data

---
## 1. Load Base Model & Processor <a id='1-load-model'></a>

We load the Thonburian Whisper medium model. This serves as both the **teacher** (for pseudo-labeling) and the **student** (for fine-tuning).

In [5]:
processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language=LANGUAGE, task=TASK, token=HF_TOKEN
)
feature_extractor = processor.feature_extractor
tokenizer = processor.tokenizer

# Use BF16 for Ampere GPUs (RTX 3090, etc.), otherwise FP32
model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=model_dtype,
    device_map="auto",
    token=HF_TOKEN,
)

# Set forced decoder IDs for Thai transcription
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task=TASK
)
# Use generation_config for generation parameters (required for transformers >= 4.46)
model.generation_config.suppress_tokens = []

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters() / 1e6:.0f}M")
print(f"Data type: {model_dtype}")

Loading weights: 100%|██████████| 480/480 [00:00<00:00, 2035.26it/s]
The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to proj_out.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded: biodatlab/whisper-th-small-combined
Parameters: 282M
Data type: torch.bfloat16


In [7]:
# ── HuggingFace Authentication ─────────────────────────────────────────────────
# Required for gated datasets like Common Voice

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("✅ Logged in to HuggingFace Hub")
else:
    print("⚠️ No HF_TOKEN set - using public models only")

✅ Logged in to HuggingFace Hub


In [9]:
# ── Load Audio Files from Local Paths ─────────────────────────────────────────────
# The 'audio' column contains file paths - load actual audio data

def load_audio_file(example):
    """Load audio file from the path specified in the 'audio' column."""
    audio_path = os.path.join(BASE_PATH, example["audio"])
    if os.path.exists(audio_path):
        # Load with librosa to get array and sampling rate
        audio_array, sr = librosa.load(audio_path, sr=SAMPLING_RATE)
        example["audio"] = {
            "array": audio_array,
            "sampling_rate": sr
        }
    else:
        # Create silent audio as fallback
        example["audio"] = {
            "array": np.zeros(SAMPLING_RATE),
            "sampling_rate": SAMPLING_RATE
        }
    return example

# Load audio files for all datasets
print("Loading audio files...")
cv_train = cv_train.map(load_audio_file, desc="Loading train audio")
cv_test = cv_test.map(load_audio_file, desc="Loading test audio")
cv_other = cv_other.map(load_audio_file, desc="Loading other audio")

# Summary
print(f"\n✅ Audio loaded successfully")
print(f"   Train: {len(cv_train)} samples")
print(f"   Test: {len(cv_test)} samples")
print(f"   Other: {len(cv_other)} samples")
if train_clean_exists:
    print(f"   ✨ Using pre-cleaned data from {TRAIN_CLEAN_PATH}")

Loading audio files...


Loading other audio: 100%|██████████| 805/805 [00:01<00:00, 553.50 examples/s]


✅ Audio loaded successfully
   Train: 23738 samples
   Test: 805 samples
   Other: 805 samples


In [10]:
# ── 3a. Thai Text Normalization ────────────────────────────────────────────────
# Adapted from Thonburian Whisper's evaluation script

def clean_thai_text(sentence: str, remove_punctuation: bool = True) -> str:
    """Normalize Thai text for ASR training/evaluation."""
    sentence = sentence.strip()

    # Remove zero-width and non-breaking spaces
    sentence = sentence.replace("\u200b", " ")
    sentence = sentence.replace("\xa0", " ")

    # Fix common Thai character errors
    sentence = sentence.replace("เเ", "แ")       # Wrong แ encoding
    sentence = sentence.replace("ํา", "ำ")        # Sara am decomposed
    sentence = sentence.replace("่ำ", "่ำ")       # Mai ek + sara am
    sentence = sentence.replace("ำ้", "้ำ")       # Wrong order: sara am before mai tho
    sentence = sentence.replace("ํ่า", "่ำ")      # Nikhahit + mai ek + sara aa

    # Normalize special characters
    sentence = sentence.replace("▁", "_")
    sentence = sentence.replace("—", "-")
    sentence = sentence.replace("–", "-")
    sentence = sentence.replace("−", "-")
    sentence = sentence.replace("\u2018", "'")
    sentence = sentence.replace("\u2019", "'")
    sentence = sentence.replace("\u201c", '"')
    sentence = sentence.replace("\u201d", '"')

    if remove_punctuation:
        sentence = "".join(
            c for c in sentence if c not in string.punctuation
        )

    return " ".join(sentence.split()).strip()


# Test it
print(clean_thai_text("เเม่ ํากิน ข้าว—ดีใจ"))

แม่ ำกิน ข้าวดีใจ


In [11]:
# ── 3b. Duration Filtering ─────────────────────────────────────────────────────

def get_duration(example):
    """Add duration_sec column."""
    example["duration_sec"] = len(example["audio"]["array"]) / example["audio"]["sampling_rate"]
    return example


def filter_by_duration(dataset, min_sec=MIN_DURATION_SEC, max_sec=MAX_DURATION_SEC, verbose=True):
    """Remove clips outside [min_sec, max_sec]."""
    dataset = dataset.map(get_duration, desc="Computing durations")
    before = len(dataset)
    dataset = dataset.filter(
        lambda x: min_sec <= x["duration_sec"] <= max_sec
    )
    after = len(dataset)
    if verbose:
        print(f"Duration filter: {before} → {after} (removed {before - after})")
    return dataset

In [ ]:
# ── 3d. Apply Full Cleaning Pipeline ──────────────────────────────────────────

def clean_dataset(dataset, apply_snr=True, verbose=True):
    """Run the full cleaning pipeline on a dataset."""
    if verbose:
        print(f"\n{'='*60}")
        print(f"Starting cleaning: {len(dataset)} samples")
        print(f"{'='*60}")

    # 1. Normalize text (no multiprocessing to avoid Jupyter issues)
    dataset = dataset.map(
        lambda x: {"sentence": clean_thai_text(x["sentence"], remove_punctuation=False)},
        desc="Normalizing text",
    )

    # Remove empty sentences
    before = len(dataset)
    dataset = dataset.filter(lambda x: len(x["sentence"].strip()) > 0)
    if verbose:
        print(f"Empty text filter: {before} → {len(dataset)}")

    # 2. Duration filter
    dataset = filter_by_duration(dataset, verbose=verbose)

    # 3. SNR filter (optional — slower)
    if apply_snr:
        dataset = filter_by_snr(dataset, verbose=verbose)

    if verbose:
        print(f"\n✅ Final clean dataset: {len(dataset)} samples")
    return dataset


# Clean the labeled training data (skip if pre-cleaned dataset was loaded)
if train_clean_exists:
    print("📋 Using pre-cleaned training data - skipping cleaning pipeline")
    cv_train_clean = cv_train
else:
    print("🧹 Applying cleaning pipeline to training data...")
    cv_train_clean = clean_dataset(cv_train, apply_snr=False, verbose=False)

🧹 Applying cleaning pipeline to training data...


Filter:   0%|          | 0/23738 [00:00<?, ? examples/s]

In [ ]:
# ── 3e. Visualize Cleaning Results ────────────────────────────────────────────

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Duration distribution
durations = cv_train_clean["duration_sec"]
axes[0].hist(durations, bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Duration (sec)")
axes[0].set_ylabel("Count")
axes[0].set_title("Audio Duration Distribution (after cleaning)")
axes[0].axvline(np.median(durations), color="red", linestyle="--", label=f"Median: {np.median(durations):.1f}s")
axes[0].legend()

# SNR distribution
if "snr_db" in cv_train_clean.column_names:
    snrs = cv_train_clean["snr_db"]
    axes[1].hist(snrs, bins=50, color="coral", edgecolor="white")
    axes[1].set_xlabel("Estimated SNR (dB)")
    axes[1].set_ylabel("Count")
    axes[1].set_title("SNR Distribution (after cleaning)")
    axes[1].axvline(np.median(snrs), color="red", linestyle="--", label=f"Median: {np.median(snrs):.1f} dB")
    axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 4a. Clean the unlabeled pool first ────────────────────────────────────────
# Apply duration + SNR filtering before spending GPU time on transcription

# Limit size for Colab — adjust as needed
MAX_PSEUDO_SAMPLES = 1000

# Select subset (cv_other already has audio loaded from cell 10)
cv_other_pool = cv_other.select(range(min(MAX_PSEUDO_SAMPLES, len(cv_other))))

# Duration filter only (SNR is slow on large sets)
cv_other_pool = cv_other_pool.map(get_duration, desc="Computing durations")
cv_other_pool = cv_other_pool.filter(
    lambda x: MIN_DURATION_SEC <= x["duration_sec"] <= MAX_DURATION_SEC
)
print(f"Pseudo-label pool after duration filter: {len(cv_other_pool)}")

In [ ]:
# ── 4b. Generate Pseudo-Labels with Confidence Scores ─────────────────────────

@torch.no_grad()
def generate_pseudo_labels(
    dataset,
    model,
    processor,
    batch_size: int = 8,
    device: str = "cuda",
):
    """
    Generate pseudo-labels with confidence scores.
    Confidence = mean log-probability of generated tokens.
    """
    model.eval()

    all_texts = []
    all_confidences = []
    
    # Get model dtype to avoid dtype mismatch
    model_dtype = next(model.parameters()).dtype

    for i in tqdm(range(0, len(dataset), batch_size), desc="Generating pseudo-labels"):
        batch = dataset[i : i + batch_size]
        audio_arrays = [a["array"] for a in batch["audio"]]

        inputs = processor(
            audio_arrays,
            sampling_rate=SAMPLING_RATE,
            return_tensors="pt",
            padding=True,
        )

        # Generate with scores - match model dtype instead of using .half()
        outputs = model.generate(
            inputs["input_features"].to(device).to(model_dtype),
            language=LANGUAGE,
            return_timestamps=False,
            output_scores=True,
            return_dict_in_generate=True,
            max_new_tokens=256,
        )

        # Decode text
        texts = processor.batch_decode(outputs.sequences, skip_special_tokens=True)
        all_texts.extend(texts)

        # Compute confidence: mean log-prob of generated tokens
        if outputs.scores:  # tuple of logits per timestep
            for seq_idx in range(len(texts)):
                token_log_probs = []
                for t, score in enumerate(outputs.scores):
                    if seq_idx < score.shape[0]:
                        log_probs = torch.log_softmax(score[seq_idx], dim=-1)
                        token_id = outputs.sequences[seq_idx, t + 1]  # +1 for decoder start token
                        if token_id < log_probs.shape[-1]:
                            token_log_probs.append(log_probs[token_id].item())

                confidence = np.exp(np.mean(token_log_probs)) if token_log_probs else 0.0
                all_confidences.append(confidence)
        else:
            all_confidences.extend([0.0] * len(texts))

    return all_texts, all_confidences


# Generate pseudo-labels
pseudo_texts, pseudo_confidences = generate_pseudo_labels(
    cv_other_pool, model, processor, batch_size=8
)

print(f"\nGenerated {len(pseudo_texts)} pseudo-labels")
print(f"Confidence stats: mean={np.mean(pseudo_confidences):.3f}, "
      f"median={np.median(pseudo_confidences):.3f}, "
      f"min={np.min(pseudo_confidences):.3f}, max={np.max(pseudo_confidences):.3f}")

In [ ]:
# ── 4c. Filter by Confidence ──────────────────────────────────────────────────

# Visualize confidence distribution
plt.figure(figsize=(8, 4))
plt.hist(pseudo_confidences, bins=50, color="mediumpurple", edgecolor="white")
plt.axvline(PSEUDO_LABEL_CONFIDENCE_THRESHOLD, color="red", linestyle="--",
            label=f"Threshold: {PSEUDO_LABEL_CONFIDENCE_THRESHOLD}")
plt.xlabel("Confidence (avg token probability)")
plt.ylabel("Count")
plt.title("Pseudo-Label Confidence Distribution")
plt.legend()
plt.show()

# Show confidence statistics before filtering
print(f"Confidence Statistics:")
print(f"  Mean:   {np.mean(pseudo_confidences):.3f}")
print(f"  Median: {np.median(pseudo_confidences):.3f}")
print(f"  Min:    {np.min(pseudo_confidences):.3f}")
print(f"  Max:    {np.max(pseudo_confidences):.3f}")
print(f"  Std:    {np.std(pseudo_confidences):.3f}")

# Show percentiles
percentiles = [10, 25, 50, 75, 90]
print(f"\nPercentiles:")
for p in percentiles:
    print(f"  {p}th:   {np.percentile(pseudo_confidences, p):.3f}")

# Filter
keep_mask = [c >= PSEUDO_LABEL_CONFIDENCE_THRESHOLD for c in pseudo_confidences]
n_kept = sum(keep_mask)

# If no samples pass threshold, suggest a lower threshold
if n_kept == 0:
    suggested_threshold = np.percentile(pseudo_confidences, 50)  # Use median as fallback
    print(f"\n⚠️ No samples passed threshold {PSEUDO_LABEL_CONFIDENCE_THRESHOLD}")
    print(f"💡 Suggested threshold (median): {suggested_threshold:.3f}")
    print(f"   Using median as threshold instead...")
    keep_mask = [c >= suggested_threshold for c in pseudo_confidences]
    n_kept = sum(keep_mask)

print(f"\nKeeping {n_kept}/{len(pseudo_confidences)} pseudo-labels "
      f"({100*n_kept/len(pseudo_confidences):.1f}%) above threshold")

In [ ]:
# ── 4d. Build Pseudo-Label Dataset ────────────────────────────────────────────

# Select kept indices
kept_indices = [i for i, keep in enumerate(keep_mask) if keep]
pseudo_dataset = cv_other_pool.select(kept_indices)

# Replace original sentences with pseudo-labels
kept_texts = [clean_thai_text(pseudo_texts[i], remove_punctuation=False) for i in kept_indices]
pseudo_dataset = pseudo_dataset.remove_columns(["sentence"])
pseudo_dataset = pseudo_dataset.add_column("sentence", kept_texts)

# Filter out empty pseudo-labels
pseudo_dataset = pseudo_dataset.filter(lambda x: len(x["sentence"].strip()) > 0)

print(f"\n✅ Pseudo-label dataset: {len(pseudo_dataset)} samples")
print("\nSample pseudo-labels:")
for i in range(min(5, len(pseudo_dataset))):
    print(f"  [{pseudo_confidences[kept_indices[i]]:.3f}] {pseudo_dataset[i]['sentence'][:80]}")

In [ ]:
# ── 4e. Combine Labeled + Pseudo-Labeled Data ─────────────────────────────────

# Ensure both datasets have the same columns
common_cols = ["audio", "sentence"]

train_labeled = cv_train_clean.select_columns(common_cols)
train_pseudo = pseudo_dataset.select_columns(common_cols)

# Concatenate
train_combined = concatenate_datasets([train_labeled, train_pseudo])
train_combined = train_combined.shuffle(seed=42)

print(f"Combined training set: {len(train_combined)} samples")
print(f"  ├── Labeled:      {len(train_labeled)}")
print(f"  └── Pseudo-label: {len(train_pseudo)}")

---
## 5. Prepare Data for Training <a id='5-prepare-training'></a>

Convert raw audio + text into Whisper's expected input format:
- Audio → log-mel spectrogram features
- Text → tokenized label IDs

In [ ]:
def prepare_dataset(example):
    """Convert a single example to model inputs."""
    audio = example["audio"]

    # Audio → mel spectrogram features
    example["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # Text → token IDs
    example["labels"] = tokenizer(example["sentence"]).input_ids

    return example


# Optionally limit samples for a quick test
if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(train_combined):
    train_combined = train_combined.select(range(MAX_TRAIN_SAMPLES))
    print(f"⚡ Quick-test mode: using {len(train_combined)} training samples")
else:
    print(f"🚀 Full training mode: using {len(train_combined)} training samples")

# ── ROBUST PROCESSING WITH CHUNKS AND CHECKPOINTING ───────────────────────
import os

# Check if we already have processed data cached
PROCESSED_TRAIN_DIR = "./processed_train_cache"
PROCESSED_EVAL_DIR = "./processed_eval_cache"

def process_in_chunks(dataset, chunk_size=5000, cache_dir=None):
    """
    Process dataset in chunks to avoid memory issues.
    Saves progress to disk so it can be resumed if interrupted.
    """
    if cache_dir and os.path.exists(cache_dir):
        print(f"📁 Found cached processed data at {cache_dir}")
        from datasets import load_from_disk
        return load_from_disk(cache_dir)
    
    total_samples = len(dataset)
    processed_chunks = []
    
    for start_idx in range(0, total_samples, chunk_size):
        end_idx = min(start_idx + chunk_size, total_samples)
        print(f"\n🔄 Processing chunk {start_idx}-{end_idx} ({end_idx}/{total_samples})")
        
        chunk = dataset.select(range(start_idx, end_idx))
        processed_chunk = chunk.map(
            prepare_dataset,
            remove_columns=chunk.column_names,
            desc=f"Processing {start_idx}-{end_idx}",
            keep_in_memory=False,  # Don't keep in memory after processing
        )
        processed_chunks.append(processed_chunk)
        
        # Free memory
        del chunk
        import gc
        gc.collect()
    
    # Concatenate all chunks
    print("\n🔗 Concatenating all chunks...")
    from datasets import concatenate_datasets
    result = concatenate_datasets(processed_chunks)
    
    # Save to disk for future runs
    if cache_dir:
        print(f"💾 Saving processed data to {cache_dir}...")
        result.save_to_disk(cache_dir)
        print(f"✅ Saved! Next run will load from cache.")
    
    return result


# Process datasets with chunking
print("Processing datasets with checkpointing...")
train_dataset = process_in_chunks(
    train_combined, 
    chunk_size=5000,  # Process 5000 samples at a time
    cache_dir=PROCESSED_TRAIN_DIR
)

# Clean test set first
cv_test_clean = clean_dataset(cv_test, apply_snr=False, verbose=False)

eval_dataset = process_in_chunks(
    cv_test_clean.select_columns(["audio", "sentence"]),
    chunk_size=1000,  # Smaller chunks for eval
    cache_dir=PROCESSED_EVAL_DIR
)

print(f"✅ Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

In [ ]:
# ── 6b. Apply LoRA (Optional) ─────────────────────────────────────────────────────
USE_LORA = True  # Set to False to disable LoRA and fine-tune all parameters

if USE_LORA:
    from peft import LoraConfig, get_peft_model
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
        bias="none",
        task_type="SEQ_2_SEQ_LM",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
else:
    print("Using full fine-tuning (all parameters trainable)")
    print(f"Trainable parameters: {model.num_parameters() / 1e6:.0f}M")

In [ ]:
# ── 6c. Evaluation Metric ─────────────────────────────────────────────────────

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    """Compute WER with Thai tokenization (deepcut) during training."""
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with pad token
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # Decode
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # Normalize Thai text
    pred_str = [clean_thai_text(p).lower() for p in pred_str]
    label_str = [clean_thai_text(l).lower() for l in label_str]

    # Tokenize with deepcut for Thai word segmentation
    pred_str = [" ".join(thai_tokenize(p, engine="deepcut")) for p in pred_str]
    label_str = [" ".join(thai_tokenize(l, engine="deepcut")) for l in label_str]

    # Filter out empty references
    pairs = [(p, l) for p, l in zip(pred_str, label_str) if len(l.strip()) > 0]
    if not pairs:
        return {"wer": 0.0}
    pred_str, label_str = zip(*pairs)

    wer = 100 * wer_metric.compute(predictions=list(pred_str), references=list(label_str))
    return {"wer": wer}

In [ ]:
# ── 6d. Training Arguments ────────────────────────────────────────────────────

# Check if BF16 is supported (RTX 3090 and newer Ampere GPUs support it)
bf16_supported = torch.cuda.is_bf16_supported()

# Adjust batch size based on training mode
if not USE_LORA:
    # Full fine-tuning: smaller batch size due to full model gradients
    train_batch_size = 4
    eval_batch_size = 4
else:
    # LoRA: can use larger batch size
    train_batch_size = 4
    eval_batch_size = 4

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",

    # Use BF16 if supported (faster, more stable), otherwise no mixed precision
    bf16=bf16_supported,
    fp16=False,  # Disable fp16 as it conflicts with float16 model dtype
    
    # Memory optimization (disable if encountering errors)
    gradient_checkpointing=False,  # Can cause issues with some setups
    
    # Evaluation & saving
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # Generation config for eval
    predict_with_generate=True,
    generation_max_length=256,

    # Logging
    logging_steps=25,
    report_to="none",  # Change to "wandb" if you use W&B

    # Misc
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_num_workers=2,
)

print(f"Training mode: {'Full fine-tuning' if not USE_LORA else 'LoRA'}")
print(f"Batch size: {train_batch_size} x {GRAD_ACCUM_STEPS} grad accum = {train_batch_size * GRAD_ACCUM_STEPS} effective")
print(f"BF16: {bf16_supported}")

In [ ]:
# ── Data Collator ─────────────────────────────────────────────────────────────

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Data collator that pads input features and labels for Whisper.
    Labels are padded with -100 so they are ignored in the loss.
    """
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Split inputs and labels
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        # Pad input features
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad labels
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding token id with -100
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Remove BOS token if appended during tokenization
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
# ── 6a. Reload Model Fresh for Training ───────────────────────────────────────
# NOTE: Due to a PEFT compatibility issue with Whisper models in some versions,
# we offer two training options:

# Option 1: Full fine-tuning (recommended for RTX 3090)
# - Works reliably with all transformers/peft versions
# - Uses BF16 mixed precision (efficient on Ampere GPUs)
# - Better convergence for domain adaptation

# Option 2: LoRA fine-tuning (experimental)
# - Uses less VRAM but has known compatibility issues
# - May not work with current PEFT versions

USE_LORA = True # Set to False to use full fine-tuning (RECOMMENDED for RTX 3090)

del model
torch.cuda.empty_cache()

# Use BF16 for better performance on Ampere GPUs (RTX 3090, etc.)
model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

if not USE_LORA:
    # Option 1: Full fine-tuning
    print(f"Loading model for full fine-tuning (dtype={model_dtype})...")
    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=model_dtype,
        device_map="auto",
        token=HF_TOKEN,
    )
    print(f"Using full fine-tuning (all parameters trainable)")
    print(f"Trainable parameters: {model.num_parameters() / 1e6:.0f}M")
else:
    # Option 2: LoRA fine-tuning (experimental)
    print("Loading model for LoRA fine-tuning...")
    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    
    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        quantization_config=quantization_config,
        device_map="auto",
        token=HF_TOKEN,
    )
    model = prepare_model_for_kbit_training(model)
    
    # Apply LoRA adapters after loading quantized model
    from peft import LoraConfig, get_peft_model
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
        bias="none",
        task_type="SEQ_2_SEQ_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task=TASK
)
# Use generation_config for generation parameters (required for transformers >= 4.46)
model.generation_config.suppress_tokens = []
model.config.use_cache = False  # Required for training

print(f"Model loaded for {'LoRA' if USE_LORA else 'full'} fine-tuning")

In [ ]:
# ── 6e. Train! ────────────────────────────────────────────────────────────────

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,  # Use processing_class instead of tokenizer
)

print("🚀 Starting training...")
trainer.train()

In [ ]:
# ── 6f. Save Fine-tuned Model ─────────────────────────────────────────────────────

if not USE_LORA:
    # Full fine-tuning: save the entire model
    model.save_pretrained(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)
    print(f"\n✅ Full fine-tuned model saved to {OUTPUT_DIR}")
else:
    # LoRA: save just the adapter (small — typically a few MB)
    model.save_pretrained(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)
    print(f"\n✅ LoRA adapter saved to {OUTPUT_DIR}")

model_size = sum(f.stat().st_size for f in __import__('pathlib').Path(OUTPUT_DIR).rglob('*') if f.is_file()) / 1e6
print(f"Model size: {model_size:.1f} MB")

---
## 7. Evaluation <a id='7-evaluation'></a>

Full evaluation on the Common Voice 13 Thai test set using the same methodology as the Thonburian Whisper paper: **deepcut tokenizer + WER/IER/SER/DER**.

In [ ]:
# ── 7a. Detailed Thai WER Computation ─────────────────────────────────────────

def compute_metrics_thai(
    pred_texts: List[str],
    ref_texts: List[str],
) -> dict:
    """
    Compute WER, IER, SER, DER using deepcut tokenizer.
    Same methodology as Thonburian Whisper evaluation.
    """
    # Normalize
    norm_preds = [clean_thai_text(p).lower() for p in pred_texts]
    norm_refs = [clean_thai_text(r).lower() for r in ref_texts]

    # Thai word segmentation
    norm_preds = [" ".join(thai_tokenize(p, engine="deepcut")) for p in tqdm(norm_preds, desc="Tokenizing preds")]
    norm_refs = [" ".join(thai_tokenize(r, engine="deepcut")) for r in tqdm(norm_refs, desc="Tokenizing refs")]

    # Filter empty references
    pairs = [(p, r) for p, r in zip(norm_preds, norm_refs) if len(r.strip()) > 0]
    norm_preds, norm_refs = zip(*pairs) if pairs else ([], [])
    norm_preds, norm_refs = list(norm_preds), list(norm_refs)

    # Compute detailed metrics
    wer_output = process_words(norm_refs, norm_preds, wer_default, wer_default)
    total_ref_words = sum(len(ref) for ref in wer_output.references)

    return {
        "WER": round(100 * wer_output.wer, 2),
        "IER (Insertion)": round(100 * wer_output.insertions / total_ref_words, 2),
        "SER (Substitution)": round(100 * wer_output.substitutions / total_ref_words, 2),
        "DER (Deletion)": round(100 * wer_output.deletions / total_ref_words, 2),
        "Total samples": len(norm_refs),
    }

In [ ]:
# ── 7b. Run Inference on Test Set ─────────────────────────────────────────────

@torch.no_grad()
def transcribe_dataset(dataset, model, processor, batch_size=8, device="cuda"):
    """Transcribe all audio in a dataset."""
    model.eval()
    
    # Get model dtype to avoid dtype mismatch
    model_dtype = next(model.parameters()).dtype
    
    all_preds = []

    for i in tqdm(range(0, len(dataset), batch_size), desc="Transcribing"):
        batch = dataset[i : i + batch_size]
        audio_arrays = [a["array"] for a in batch["audio"]]

        inputs = processor(
            audio_arrays,
            sampling_rate=SAMPLING_RATE,
            return_tensors="pt",
            padding=True,
        )

        # Prepare inputs - use 'input_features' keyword for Whisper/PEFT compatibility
        input_features = inputs["input_features"].to(device).to(model_dtype)
        
        # For Whisper + PEFT, use input_features as keyword argument
        pred_ids = model.generate(
            input_features=input_features,
            language=LANGUAGE,
            return_timestamps=False,
            max_new_tokens=256,
        )

        preds = processor.batch_decode(pred_ids, skip_special_tokens=True)
        all_preds.extend(preds)

    return all_preds

In [ ]:
# Prepare test set with audio
test_with_audio = cv_test_clean.select_columns(["audio", "sentence"])

# Run transcription
predictions = transcribe_dataset(test_with_audio, model, processor, batch_size=8)
references = test_with_audio["sentence"]

# Compute metrics
results = compute_metrics_thai(predictions, references)

print("\n" + "="*50)
print("📊 EVALUATION RESULTS (Fine-tuned Model)")
print("="*50)
for k, v in results.items():
    print(f"  {k}: {v}")

In [ ]:
# ── 7c. Compare with Base Model (Optional) ───────────────────────────────────
# Uncomment to run — takes extra time

# Load base model for comparison
base_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto", token=HF_TOKEN
)
base_model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task=TASK
)
base_predictions = transcribe_dataset(test_with_audio, base_model, processor)
base_results = compute_metrics_thai(base_predictions, references)

print("\n" + "="*50)
print("📊 COMPARISON")
print("="*50)
print(f"  Base model WER:      {base_results['WER']}")
print(f"  Fine-tuned model WER: {results['WER']}")
print(f"  Improvement:          {base_results['WER'] - results['WER']:.2f} absolute")

In [ ]:
print("\n" + "="*50)
print("📊 COMPARISON")
print("="*50)
print(f"  Base model WER:      {base_results['WER']}")
print(f"  Fine-tuned model WER: {results['WER']}")
print(f"  Improvement:          {base_results['WER'] - results['WER']:.2f} absolute")

In [ ]:
# ── 7d. Inspect Predictions ──────────────────────────────────────────────────

# Show some examples
n_examples = 10
df = pd.DataFrame({
    "Reference": references[:n_examples],
    "Prediction": predictions[:n_examples],
})
pd.set_option("display.max_colwidth", 80)
df

---
## 8. Inference & Export <a id='8-inference'></a>

Use the fine-tuned model for inference, or push it to HuggingFace Hub.

In [ ]:
# ── 8a. Quick Inference with the Fine-tuned Model ────────────────────────────

from transformers import pipeline

# Determine the dtype based on model
model_dtype = model.dtype if hasattr(model, 'dtype') else torch.bfloat16

# Load pipeline with the fine-tuned model
pipe = pipeline(
    task="automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    device=0 if torch.cuda.is_available() else "cpu",
    torch_dtype=model_dtype,
)

# Transcribe a sample from the test set
sample = cv_test_clean[0]
# Convert audio to the correct dtype (handle both list and numpy array)
audio_array = np.array(sample["audio"]["array"], dtype=np.float32)

result = pipe(
    audio_array,
    generate_kwargs={"language": "<|th|>", "task": "transcribe"},
)
print(f"Reference:  {sample['sentence']}")
print(f"Prediction: {result['text']}")

In [ ]:
# ── 8b. Export Model for Deployment ────────────────────────────────────────────────

if not USE_LORA:
    # Full fine-tuning: model is already complete, just copy to export directory
    MERGED_DIR = "./whisper-th-finetuned-export"
    import shutil
    shutil.copytree(OUTPUT_DIR, MERGED_DIR, dirs_exist_ok=True)
    print(f"✅ Full fine-tuned model exported to {MERGED_DIR}")
else:
    # LoRA: Merge adapter weights into base model for deployment
    MERGED_DIR = "./whisper-th-finetuned-merged"
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(MERGED_DIR)
    processor.save_pretrained(MERGED_DIR)
    print(f"✅ Merged model saved to {MERGED_DIR}")

In [ ]:
# ── 8c. Push to HuggingFace Hub (Optional) ───────────────────────────────────

# Uncomment and set your repo name:
# HUB_REPO = "your-username/whisper-th-medium-domain-finetuned"
#
# merged_model.push_to_hub(HUB_REPO)
# processor.push_to_hub(HUB_REPO)
#
# print(f"✅ Model pushed to https://huggingface.co/{HUB_REPO}")